# Phase 12A — SAR Training (SQL)

**SAR = Schema-Aware Retriever** — a small retrieval model (~16M params) that finds the most structurally similar past examples for a given question.

## What this notebook trains

| | Detail |
|---|---|
| **Model trained** | `SchemaAwareModel` (~16M params) — custom cross-attention network |
| **Model NOT trained** | `BGE` (BAAI/bge-large-en-v1.5, 330M params) — used frozen as text encoder |
| **Input data** | `Data/rag_corpus/spider_sql_rag.json` — 7000 (question, SQL, structural_type) entries |
| **Training method** | Triplet contrastive loss on SQL structural type |
| **Output** | `sar_model.pt` (~60 MB) saved to Google Drive |
| **Est. training time** | ~25–30 minutes on T4 |

## How it works

1. BGE encodes every question into a 1024-dim vector (runs once, cached to disk)
2. Questions are grouped by their SQL structural type (57 unique types across 7000 entries)
3. For each question, a triplet is formed:
   - **Anchor**: the question
   - **Positive**: another question with the **same** structural type (e.g., both need GROUP BY)
   - **Negative**: a question with a **different** structural type (e.g., simple SELECT)
4. Triplet loss trains the SchemaAwareModel to pull same-structure questions together and push different-structure ones apart
5. After training, at inference time SAR retrieves the top-3 most structurally similar past examples to use as few-shot examples for the Generator

## Why this matters

Without SAR, the Generator gets random or keyword-matched examples. With SAR, it gets examples that needed the **same SQL pattern** — dramatically improving generation accuracy for complex queries (subqueries, GROUP BY + HAVING, multi-join).

## Cell 1 — Verify GPU

In [ ]:
import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
print(result.stdout)

gpu = result.stdout
if 'T4' in gpu:
    print('T4 detected — SAR trains in ~25-30 min on T4 ✅')
elif 'A100' in gpu:
    print('A100 detected — SAR will be faster (~10 min) ✅')
else:
    print('⚠️  Unknown GPU — check Runtime → Change runtime type')

## Cell 2 — Mount Google Drive

The trained model (`sar_model.pt`) is saved to Drive so it survives session disconnects.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

OUT_DIR = '/content/drive/MyDrive/codegen/checkpoints/sar_sql'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output dir: {OUT_DIR} ✅')

## Cell 3 — Clone or update repo

In [ ]:
%%bash
set -e

REPO_DIR="/content/Codegen"
BRANCH="phase/12a-sar-sql"

if [ -d "$REPO_DIR/.git" ]; then
    echo "Repo exists — pulling latest"
    cd "$REPO_DIR"
    git fetch origin
    git checkout $BRANCH
    git pull origin $BRANCH
else
    echo "Cloning repo"
    git clone https://github.com/kethansplunk/Codegen.git "$REPO_DIR"
    cd "$REPO_DIR"
    git checkout $BRANCH
fi

echo "Branch : $(git branch --show-current)"
echo "Commit : $(git log --oneline -1)"

## Cell 4 — Install dependencies

- **FlagEmbedding**: loads the BGE model to encode questions into 1024-dim vectors
- **scikit-learn**: used internally for cosine similarity during evaluation
- **torch**: already installed on Colab, but pinned to avoid version conflicts

In [ ]:
!pip install -q FlagEmbedding scikit-learn
print('Dependencies installed ✅')

## Cell 5 — Inspect training corpus

The SAR corpus is `spider_sql_rag.json` — built in Phase 5A by parsing the ground-truth SQL for every Spider training question.

Each entry has:
- `question`: natural language question
- `sql`: the ground-truth SQL query
- `structural_type`: a 6-dim vector describing the SQL pattern (joins, GROUP BY, ORDER BY, etc.)

SAR groups questions by `structural_type` and trains on triplets — no schema linking needed.

In [ ]:
import json
from collections import Counter

CORPUS_PATH = '/content/Codegen/Data/rag_corpus/spider_sql_rag.json'

with open(CORPUS_PATH) as f:
    corpus = json.load(f)

print(f'Total entries     : {len(corpus)}')
print(f'Keys per entry    : {list(corpus[0].keys())}')
print(f'Sample question   : {corpus[0]["question"]}')
print(f'Sample SQL        : {corpus[0]["sql"]}')
print(f'Structural type   : {corpus[0]["structural_type"]}')

# Count unique structural types — these define how triplets are formed
def st_key(st): return tuple(sorted(st.items()))
counts = Counter(st_key(d['structural_type']) for d in corpus)
print(f'\nUnique structural types : {len(counts)}')
print('Top 5 most common types:')
for k, v in counts.most_common(5):
    d = dict(k)
    flags = [name for name, val in d.items() if val is True]
    joins = d.get('num_joins', 0)
    desc  = f"{v:4d} entries | joins={joins}" + (f" | {', '.join(flags)}" if flags else " | simple SELECT")
    print(f'  {desc}')

## Cell 6 — Run SAR training

### What happens inside

**Phase A — BGE encoding (~2 min, runs once)**
- Downloads `BAAI/bge-large-en-v1.5` (~1.3 GB) on first run
- Encodes all 7000 questions into 1024-dim vectors in batches of 64
- Saves embeddings to `Data/sar_emb_cache.pkl` — if the cell is re-run, this step is skipped

**Phase B — Triplet training (~20-25 min, 10 epochs)**
- Builds (anchor, positive, negative) triplets from the 57 structural types
- Trains the SchemaAwareModel with triplet margin loss (margin=0.3)
- Prints average loss after each epoch — loss should decrease from ~0.3 down to ~0.05
- Saves `sar_model.pt` to Google Drive at the end

### BGE is NOT trained — only SchemaAwareModel is

BGE weights are frozen throughout. Only the ~16M parameter SchemaAwareModel
(4 linear layers + 2 attention layers) is updated by the optimizer.

> **If Colab disconnects:** Re-run Cells 2-4, then re-run this cell.
> BGE encoding takes ~2 min to redo (cache is in `/content/Codegen/Data/`).
> Training restarts from scratch — but 25 min total means losing at most a few epochs.

In [ ]:
%%bash
cd /content/Codegen

echo "=== Phase A: BGE embedding (cached after first run) ==="
echo "=== Phase B: SchemaAwareModel triplet training        ==="
echo ""

python -m src.sar.train \
    --corpus Data/rag_corpus/spider_sql_rag.json \
    --out    /content/drive/MyDrive/codegen/checkpoints/sar_sql \
    --epochs 10 \
    --batch  32 \
    --lr     1e-4 \
    --margin 0.3

## Cell 7 — Verify output

After training completes, `sar_model.pt` should be ~60 MB in your Drive checkpoint directory.

At inference time, `SARRetriever` in `src/sar/infer.py` loads this file alongside BGE (which is re-downloaded from HuggingFace) to serve retrieval queries.

In [ ]:
import os

OUT_DIR = '/content/drive/MyDrive/codegen/checkpoints/sar_sql'

print('Files in output directory:')
for fname in sorted(os.listdir(OUT_DIR)):
    fpath    = os.path.join(OUT_DIR, fname)
    size_mb  = os.path.getsize(fpath) / 1e6
    print(f'  {fname:<40} {size_mb:>8.1f} MB')

model_path = os.path.join(OUT_DIR, 'sar_model.pt')
if os.path.exists(model_path):
    size = os.path.getsize(model_path) / 1e6
    print(f'\nsar_model.pt size: {size:.1f} MB  (expected ~60 MB) ✅')
    print('SAR SQL training complete — ready for Phase 12B (NoSQL) or Generator training')
else:
    print('\n⚠️  sar_model.pt not found — training may still be running or failed')

## Cell 8 — Quick retrieval smoke test (optional)

Loads the trained model and retrieves the top-3 most structurally similar examples for a sample question.
Run this after Cell 7 confirms `sar_model.pt` exists.

In [ ]:
import sys
sys.path.insert(0, '/content/Codegen')

from src.sar.infer import SARRetriever

MODEL_PATH  = '/content/drive/MyDrive/codegen/checkpoints/sar_sql/sar_model.pt'
CORPUS_PATH = '/content/Codegen/Data/rag_corpus/spider_sql_rag.json'

retriever = SARRetriever(
    model_path=MODEL_PATH,
    corpus_path=CORPUS_PATH,
)

# Test with a GROUP BY question — should retrieve other GROUP BY questions
test_question = "How many singers are there in each country?"
results = retriever.retrieve(test_question, top_k=3)

print(f'Query: "{test_question}"')
print(f'\nTop-3 structurally similar examples:')
for i, r in enumerate(results, 1):
    print(f'\n  [{i}] Question : {r["question"]}')
    print(f'       SQL      : {r["sql"]}')
    print(f'       Type     : {r["structural_type"]}')